# LFM2 NL2Bash logit distillation (A100)

This notebook distills `LiquidAI/LFM2.5-2.6B` into `LiquidAI/LFM2-350M`, then reloads the result in NF4/Q4 for evaluation. Select **Runtime → Change runtime type → A100 GPU** before starting.

The models have incompatible vocabularies (65,536 vs. 128,000 tokens). The training script therefore transplants the student's learned embedding and LM-head rows into the teacher tokenizer wherever token strings match. This makes exact full-vocabulary KL valid. Because LFM2-350M has untied input/output embeddings, the expanded vocabulary makes the resulting small model about 480M parameters, although its transformer body is unchanged.

The loss is response-only `0.3 × gold CE + 0.7 × KL(teacher || student)` at temperature 2.0. Teacher logits are computed online under the gold Bash trajectory, so no hundreds-of-gigabytes logit cache is needed.

In [ ]:
!nvidia-smi
!pip -q install --upgrade -r https://raw.githubusercontent.com/micrictor/shellai/training/requirements-distill.txt
!python -c "import transformers; print('transformers', transformers.__version__)"

In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
repo = Path('/content/shellai')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'training', '--single-branch', 'https://github.com/micrictor/shellai.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)

# Optional: create a Colab secret named HF_TOKEN for authenticated downloads
# and another named HF_REPO_ID (for example, username/lfm2-nl2bash-distilled)
# if you want the final checkpoint pushed to the Hub.
try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
print('Working directory:', Path.cwd())

## 1. Smoke test

This uses only 64 training examples. It verifies model loading, vocabulary transplantation, forward/backward passes, checkpoint writing, and validation before committing an A100 session to the full run.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    smoke_command = [
        'python', 'train_lfm_distill.py',
        '--output-dir', '/content/lfm-distill-smoke',
        '--epochs', '1',
        '--max-train-samples', '64',
        '--max-validation-samples', '32',
        '--batch-size', '2',
        '--gradient-accumulation-steps', '4',
        '--logging-steps', '2',
        '--no-save-every-epoch',
    ]
    subprocess.run(smoke_command, check=True)

## 2. Full distillation run

Defaults: two epochs, micro-batch 2, gradient accumulation 8 (effective batch 16), and 256-token sequences. `checkpoint-last` is replaced after each epoch so optimizer state does not consume all of Drive; `final` is a clean inference checkpoint. If a 40 GB A100 runs out of memory, change batch size to 1 and accumulation to 16. The official 300-row NL2SH-ALFA test configuration is never loaded during training.

In [ ]:
RUN_FULL_TRAINING = True
OUTPUT_DIR = '/content/drive/MyDrive/shellai/lfm2-nl2bash-distilled'
RESUME_FROM = None  # set to OUTPUT_DIR + '/checkpoint-last' after an interruption

if RUN_FULL_TRAINING:
    command = [
        'python', 'train_lfm_distill.py',
        '--output-dir', OUTPUT_DIR,
        '--epochs', '2',
        '--batch-size', '2',
        '--gradient-accumulation-steps', '8',
        '--max-length', '256',
        '--temperature', '2.0',
        '--distill-weight', '0.7',
        '--learning-rate', '2e-5',
        '--logging-steps', '10',
    ]
    if RESUME_FROM:
        command += ['--resume-from', RESUME_FROM]
    try:
        hub_repo = userdata.get('HF_REPO_ID')
    except Exception:
        hub_repo = None
    if hub_repo:
        command += ['--push-to-hub', hub_repo]
    print(' '.join(command))
    subprocess.run(command, check=True)

## 3. Q4 evaluation against the original student

This runtime-quantizes each model with bitsandbytes NF4, evaluates deterministic generation on the isolated official test set, and unloads one model before loading the next. It never executes generated commands. Exact match is strict; token F1 and simple syntax balance are more informative during iteration.

In [ ]:
import gc, re, time
from collections import Counter
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SYSTEM = ('You translate natural-language requests into exactly one Bash command. '
          'Return only the command, with no Markdown, explanation, or alternatives.')

def make_prompt(nl):
    return ('<|startoftext|><|im_start|>system\n' + SYSTEM + '<|im_end|>\n'
            '<|im_start|>user\nGenerate one Bash command: ' + nl.strip() + '<|im_end|>\n'
            '<|im_start|>assistant\n<think></think>\n')

def clean_answer(text):
    text = text.split('<|im_end|>', 1)[0].strip()
    fenced = re.search(r'```(?:bash|sh)?\s*(.*?)```', text, re.I | re.S)
    return (fenced.group(1) if fenced else text).strip()

def token_f1(reference, candidate):
    ref, pred = reference.split(), candidate.split()
    if not ref or not pred: return 0.0
    overlap = sum((Counter(ref) & Counter(pred)).values())
    if not overlap: return 0.0
    p, r = overlap / len(pred), overlap / len(ref)
    return 2 * p * r / (p + r)

def syntax_balanced(command):
    return (bool(command) and command.count(chr(34)) % 2 == 0
            and command.count(chr(39)) % 2 == 0
            and command.count('(') == command.count(')')
            and command.count('[') == command.count(']'))

def evaluate_q4(model_id, samples):
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                  bnb_4bit_use_double_quant=True,
                                  bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=quant, device_map={'': 0}, dtype=torch.bfloat16)
    predictions, latencies = [], []
    for row in samples:
        encoded = tokenizer(make_prompt(row['nl']), return_tensors='pt').to('cuda')
        torch.cuda.synchronize(); started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**encoded, max_new_tokens=96, do_sample=False,
                                       eos_token_id=tokenizer.eos_token_id,
                                       pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize(); latencies.append(1000 * (time.perf_counter() - started))
        continuation = generated[0, encoded['input_ids'].shape[1]:]
        predictions.append(clean_answer(tokenizer.decode(continuation, skip_special_tokens=False)))
    references = [[row['bash']] + ([row['bash2']] if row.get('bash2') else []) for row in samples]
    result = {
        'model': str(model_id),
        'examples': len(samples),
        'exact_match_pct': 100 * np.mean([p in rs for p, rs in zip(predictions, references)]),
        'token_f1': np.mean([max(token_f1(r, p) for r in rs) for p, rs in zip(predictions, references)]),
        'syntax_balanced_pct': 100 * np.mean([syntax_balanced(p) for p in predictions]),
        'latency_ms': np.mean(latencies),
    }
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    return result, predictions[:5]

test = load_dataset('westenfelder/NL2SH-ALFA', 'test', split='train').select(range(100))
results = []
for model_id in ['LiquidAI/LFM2-350M', OUTPUT_DIR + '/final']:
    result, examples = evaluate_q4(model_id, test)
    results.append(result)
    print(result)
    print('First five predictions:', examples)
results

## Reading the result

Do not select a checkpoint from training loss alone. Prefer lower held-out KL **and** improved official-test token F1/exact match after Q4 loading. If the student imitates teacher quirks while command accuracy falls, rerun with `--distill-weight 0.5`. If both validation CE and KL are still falling at epoch 2, resume for one additional epoch at `1e-5`. Keep the official test set out of all such choices; for repeated tuning, use the validation split and run the official test once on the chosen configuration.